In [1]:
# importar librerias
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


In [2]:
# rutas base
REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.today().strftime('%Y%m%d')
INPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '01_lectura_v2' / RUN_ID
OUTPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '02_train_test_v2' / RUN_ID
REPORTS_DIR = REPO_ROOT / 'reports' / '02_train_test_v2' / RUN_ID

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

input_file = INPUT_DIR / 'df_triage_encoded_v2.parquet'
if not input_file.exists():
    candidates = sorted((REPO_ROOT / 'data' / 'intermediate' / '01_lectura_v2').glob('*/df_triage_encoded_v2.parquet'))
    if candidates:
        input_file = candidates[-1]
        print(f'Usando ultimo parquet encontrado: {input_file}')
    else:
        raise FileNotFoundError('No se encontro df_triage_encoded_v2.parquet en data/intermediate/01_lectura_v2')

BINARY_TARGETS = {
    '1_vs_resto': 1,
    '5_vs_resto': 5,
}

print('Rutas configuradas:')
print(f'INPUT_FILE: {input_file}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'REPORTS_DIR: {REPORTS_DIR}')
print(f'TARGETS: {list(BINARY_TARGETS.keys())}')


Rutas configuradas:
INPUT_FILE: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\01_lectura_v2\20260329\df_triage_encoded_v2.parquet
OUTPUT_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\02_train_test_v2\20260329
REPORTS_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\02_train_test_v2\20260329
TARGETS: ['1_vs_resto', '5_vs_resto']


In [3]:
df = pd.read_parquet(input_file)

X = df.drop(columns=['nivel_triage'])
y_original = df['nivel_triage']

total_samples = len(df)
n_features = X.shape[1]


In [4]:
for target_name, target_value in BINARY_TARGETS.items():
    output_subdir = OUTPUT_DIR / target_name
    reports_subdir = REPORTS_DIR / target_name
    output_subdir.mkdir(parents=True, exist_ok=True)
    reports_subdir.mkdir(parents=True, exist_ok=True)

    y = (y_original == target_value).astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # guardar archivos
    X_train.to_parquet(output_subdir / 'X_train.parquet')
    X_test.to_parquet(output_subdir / 'X_test.parquet')
    y_train.to_frame(name='nivel_triage').to_parquet(output_subdir / 'y_train.parquet')
    y_test.to_frame(name='nivel_triage').to_parquet(output_subdir / 'y_test.parquet')

    # calcular metricas descriptivas
    class_dist_total = y.value_counts(normalize=True).sort_index().mul(100).round(2).to_dict()
    class_dist_train = y_train.value_counts(normalize=True).sort_index().mul(100).round(2).to_dict()
    class_dist_test = y_test.value_counts(normalize=True).sort_index().mul(100).round(2).to_dict()

    metrics = {
        'target_name': target_name,
        'target_positive_class': int(target_value),
        'target_definition': f'{target_value} vs resto',
        'total_samples': total_samples,
        'n_features': n_features,
        'class_distribution_percent': class_dist_total,
        'train_distribution_percent': class_dist_train,
        'test_distribution_percent': class_dist_test,
    }

    with open(reports_subdir / 'split_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=4)

    print(f'\nMetricas del split: {target_name}')
    print(f'Total de muestras: {total_samples}')
    print(f'Total de features: {n_features}')
    print(f'Definicion del target: {target_value} vs resto')
    print('Distribucion de clases (%):')
    print('  Clase | Total | Train | Test')
    print('  ------|-------|-------|------')
    for c in sorted(class_dist_total):
        total = class_dist_total.get(c, 0)
        train = class_dist_train.get(c, 0)
        test = class_dist_test.get(c, 0)
        print(f'   {c:<5} | {total:>5.1f}% | {train:>5.1f}% | {test:>5.1f}%')

    print(f'Metricas guardadas en: {reports_subdir / "split_metrics.json"}')
    print(f'Splits guardados en: {output_subdir}')



Metricas del split: 1_vs_resto
Total de muestras: 560486
Total de features: 571
Definicion del target: 1 vs resto
Distribucion de clases (%):
  Clase | Total | Train | Test
  ------|-------|-------|------
   0     |  90.7% |  90.7% |  90.7%
   1     |   9.3% |   9.3% |   9.3%
Metricas guardadas en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\02_train_test_v2\20260329\1_vs_resto\split_metrics.json
Splits guardados en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\02_train_test_v2\20260329\1_vs_resto

Metricas del split: 5_vs_resto
Total de muestras: 560486
Total de features: 571
Definicion del target: 5 vs resto
Distribucion de clases (%):
  Clase | Total | Train | Test
  ------|-------|-------|------
   0     |  72.3% |  72.3% |  72.3%
   1     |  27.7% |  27.7% |  27.7%
Metricas guardadas en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\02_train_test_v2\20260329\5_vs_resto\split_metrics.json
Split